In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd


@dataclass(frozen=True)
class Block:
    sid: str
    block_id: int
    X: np.ndarray
    y: np.ndarray


class SleepDataset:
    """
    Minimal dataset for future sleep stage prediction on precomputed per-epoch features.

    Assumptions:
      - Each subject has a Parquet file with one row per epoch.
      - Parquet contains at least: epoch_id (or implicit row order) and a stage column.
      - Features are all other numeric columns (except excluded columns).
    """

    def __init__(
        self,
        features_dir: Union[str, Path],
        *,
        stage_col: str = "sleep_stage",
        epoch_id_col: Optional[str] = "epoch_id",
        exclude_cols: Optional[Sequence[str]] = None,
        metadata_df: Optional[pd.DataFrame] = None,
        metadata_key: str = "sid",
        block_len: int = 1080,  # 1.5h if epoch_sec=5
        lookback_epochs: int = 5,  # 25s/5s = 5 (from your extended_time)
        drop_stage_values: Optional[Sequence[str]] = None,
        sort_by_epoch_id: bool = True,
    ):
        self.features_dir = Path(features_dir)
        self.stage_col = stage_col
        self.epoch_id_col = epoch_id_col
        self.block_len = int(block_len)
        self.lookback_epochs = int(lookback_epochs)
        self.sort_by_epoch_id = sort_by_epoch_id

        self.exclude_cols = set(exclude_cols or [])
        # Always exclude label/time-ish columns if present
        self.exclude_cols |= {stage_col}
        if epoch_id_col:
            self.exclude_cols |= {epoch_id_col}
        self.exclude_cols |= {"epoch_start", "epoch_end"}  # harmless if absent

        self.metadata_df = metadata_df
        self.metadata_key = metadata_key

        self.drop_stage_values = set(drop_stage_values or [])

        # Index available subjects
        self._sid_to_path: Dict[str, Path] = {}
        for p in sorted(self.features_dir.glob("*.parquet")):
            sid = p.stem
            self._sid_to_path[sid] = p

        if not self._sid_to_path:
            raise FileNotFoundError(f"No .parquet files found in: {self.features_dir}")

    @property
    def subjects(self) -> List[str]:
        return sorted(self._sid_to_path.keys())

    def loso_splits(self) -> Iterator[Tuple[str, List[str], List[str]]]:
        """
        Yields (test_sid, train_sids, test_sids) for leave-one-subject-out.
        """
        sids = self.subjects
        for test_sid in sids:
            train_sids = [s for s in sids if s != test_sid]
            yield test_sid, train_sids, [test_sid]

    def _load_subject_df(self, sid: str) -> pd.DataFrame:
        path = self._sid_to_path[sid]
        df = pd.read_parquet(path)

        # Optional: drop unwanted stage values (e.g., 'P')
        if self.drop_stage_values and self.stage_col in df.columns:
            df = df[~df[self.stage_col].isin(self.drop_stage_values)].reset_index(drop=True)

        # Sort if epoch_id exists
        if self.sort_by_epoch_id and self.epoch_id_col and self.epoch_id_col in df.columns:
            df = df.sort_values(self.epoch_id_col, kind="mergesort").reset_index(drop=True)

        # Join metadata (constant per subject)
        if self.metadata_df is not None:
            if self.metadata_key not in self.metadata_df.columns:
                raise ValueError(f"metadata_df must contain column '{self.metadata_key}'")
            row = self.metadata_df[self.metadata_df[self.metadata_key] == sid]
            if len(row) != 1:
                raise ValueError(f"Expected exactly one metadata row for sid={sid}, found {len(row)}")
            meta = row.iloc[0].drop(labels=[self.metadata_key])
            # add meta columns as constants
            for c, v in meta.items():
                df[c] = v

        return df

    def _infer_feature_cols(self, df: pd.DataFrame) -> List[str]:
        # Keep only numeric columns and not excluded
        cols = []
        for c in df.columns:
            if c in self.exclude_cols:
                continue
            if pd.api.types.is_numeric_dtype(df[c]):
                cols.append(c)
        if not cols:
            raise ValueError("No numeric feature columns found after exclusions.")
        return cols

    def iter_blocks(
        self,
        sids: Sequence[str],
        *,
        horizon_k: int,
        purge: bool = True,
        feature_cols: Optional[Sequence[str]] = None,
        drop_na_labels: bool = True,
    ) -> Iterator[Block]:
        """
        Stream blocks for given subjects.

        purge=True applies:
          - drop first lookback_epochs of each block
          - drop last horizon_k of each block (labels become NaN after shift)
        """
        k = int(horizon_k)
        if k < 0:
            raise ValueError("horizon_k must be >= 0")

        for sid in sids:
            df = self._load_subject_df(sid)
            if self.stage_col not in df.columns:
                raise ValueError(f"Missing stage_col='{self.stage_col}' in {sid}")

            # Apply horizon shift AFTER loading (and conceptually after split)
            # y[t] = stage[t + k]  => shift(-k)
            y = df[self.stage_col].shift(-k)

            # Determine feature cols once per subject
            cols = list(feature_cols) if feature_cols is not None else self._infer_feature_cols(df)

            X_df = df[cols]

            # Optional: drop rows where y is NaN (typically last k rows)
            if drop_na_labels:
                mask = ~y.isna()
                X_df = X_df[mask].reset_index(drop=True)
                y = y[mask].reset_index(drop=True)

            X_all = X_df.to_numpy(dtype=np.float32, copy=False)
            y_all = y.to_numpy()

            n = len(y_all)
            if n < self.block_len:
                continue  # subject too short for even one block

            n_blocks = n // self.block_len  # full blocks only
            for b in range(n_blocks):
                start = b * self.block_len
                end = start + self.block_len

                Xb = X_all[start:end]
                yb = y_all[start:end]

                if purge:
                    # purge at the START: lookback window could cross boundaries
                    if self.lookback_epochs > 0:
                        Xb = Xb[self.lookback_epochs:]
                        yb = yb[self.lookback_epochs:]

                    # purge at the END: avoid horizon contamination near end-of-block
                    # (even after global drop_na_labels, this protects block edges cleanly)
                    if k > 0:
                        Xb = Xb[:-k]
                        yb = yb[:-k]

                if len(yb) == 0:
                    continue

                yield Block(sid=sid, block_id=b, X=Xb, y=yb)

    def load_all_blocks(
        self,
        sids: Sequence[str],
        *,
        horizon_k: int,
        purge: bool = True,
        feature_cols: Optional[Sequence[str]] = None,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Materialize all blocks into arrays for classic tabular training.

        Returns:
          X: (N, n_features)
          y: (N,)
          groups: (N,) subject ids encoded as integers (useful for GroupKFold)
        """
        X_list: List[np.ndarray] = []
        y_list: List[np.ndarray] = []
        g_list: List[np.ndarray] = []

        sid_to_gid = {sid: i for i, sid in enumerate(sorted(set(sids)))}

        for blk in self.iter_blocks(
            sids,
            horizon_k=horizon_k,
            purge=purge,
            feature_cols=feature_cols,
        ):
            X_list.append(blk.X)
            y_list.append(blk.y)
            g_list.append(np.full(len(blk.y), sid_to_gid[blk.sid], dtype=np.int32))

        if not X_list:
            raise ValueError("No blocks produced. Check block_len, horizon_k, and data length.")

        X = np.vstack(X_list)
        y = np.concatenate(y_list)
        groups = np.concatenate(g_list)
        return X, y, groups

In [4]:
features_dir = "./data_feat/"  # contains S002.parquet, S003.parquet, ...
meta = pd.read_csv("participant_info.csv")  # must include column 'sid' or rename

ds = SleepDataset(
    features_dir,
    stage_col="sleep_stage",
    epoch_id_col="epoch_id",
    metadata_df=meta.rename(columns={"SID": "sid"}),  # if needed
    metadata_key="sid",
    block_len=1080,          # 1.5 hours
    lookback_epochs=6,       # 30s / 5s
    drop_stage_values=["P"], # optional
)

In [5]:
ds

In [ ]:
from sklearn.metrics import f1_score
from xgboost import XGBClassifier  # or RandomForestClassifier

k = 1  # horizon in epochs

for test_sid, train_sids, test_sids in ds.loso_splits():
    X_train, y_train, _ = ds.load_all_blocks(train_sids, horizon_k=k)
    X_test, y_test, _ = ds.load_all_blocks(test_sids, horizon_k=k)

    clf = XGBClassifier(
        max_depth=6,
        n_estimators=400,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="mlogloss",
    )
    clf.fit(X_train, y_train)

    pred = clf.predict(X_test)
    print(test_sid, f1_score(y_test, pred, average="macro"))